[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C59_VLA_Perception_Interface_Course/02_action_repr/02_action_representation.ipynb)

# 02 · 动作表示与输出头设计（分箱 / 多模态失败 / 流匹配 / chunking / 频率账）

目标：用**可运行的数值**把「为什么回归头会输出一个撞向隔离墩的动作」钉死，
并把 chunking 与分层控制的频率账算到能直接写进设计文档。

本 notebook 你会亲手实现：
1. **三种动作分箱**（均匀 / 分位数 / μ-law）与它们按工况分桶的重建误差
2. **多模态动作分布上的回归失败**：$\ell_2$ 收敛到均值、$\ell_1$ 收敛到中位数，两者都撞墩
3. **「加数据 / 加容量 / 加噪声」三种错误补救**的失效证明
4. **闭式速度场的流匹配采样器**，并验证一个漂亮的事实：**1 步 Euler = 回归**
5. **轨迹参数化**：多项式阶数 vs 拟合误差；以及「输出控制量会把车型烧进权重」的量化
6. **action chunking + temporal ensembling**：调用次数、RMSE、jerk 三者的取舍
7. **$(K, H, \Delta t)$ 的四约束求解器**，以及与 $\Delta t$ 无关的可行性条件 $2T_{\text{infer}}+\ldots\le T_{\text{budget}}$

> 心智模型：**输出头要给的是分布，不是均值。均值在多模态下恰好是最危险的那个答案。**

## 1 · 动作分箱：均匀 / 分位数 / μ-law

先合成一份物理上合理的前轮转角数据：**82% 的时间在 ±1° 内（高速直行微调），
其余是大转角（路口、掉头）**。这是真实驾驶数据的典型形状——极度重尾。

In [ ]:
import numpy as np, math
rng = np.random.default_rng(0)

STEER_MAX = 35.0          # 前轮转角上限（度）
WHEELBASE = 2.9           # 轴距（米）

def make_steer(n, rng):
    """合成规则：90% 抽自 N(0, 0.6°)（直行微调），10% 抽自 N(0, 12°)（转弯），再截断到 ±35°。"""
    z = rng.random(n)
    small = rng.normal(0, 0.6, n)
    large = rng.normal(0, 12.0, n)
    return np.clip(np.where(z < 0.90, small, large), -STEER_MAX, STEER_MAX)

A = make_steer(200_000, rng)
straight = np.abs(A) < 1.0                      # 「直行工况」：|δ| < 1°
print(f'样本数 {len(A)}   直行工况占比 {straight.mean():.1%}   '
      f'|δ|>10° 占比 {(np.abs(A)>10).mean():.2%}')
assert straight.mean() > 0.75, '合成数据应当极度集中在 0 附近'
print('✅ 动作分布是重尾的：绝大多数时间在做微调，极值只在少数工况出现')

In [ ]:
def uniform_edges(lo, hi, B):
    return np.linspace(lo, hi, B + 1)

def quantile_edges(a, B):
    """分位数分箱：每个 bin 承载相同的概率质量（= 先做概率积分变换再均匀分箱）。"""
    e = np.quantile(a, np.linspace(0, 1, B + 1))
    e[0] = min(e[0], a.min()); e[-1] = max(e[-1], a.max())
    return np.unique(e)

def mulaw_edges(lo, hi, B, mu=255.0):
    """μ-law 压扩：中心密、边缘疏，形状由超参 mu 决定而**不由数据决定**（可复现性好）。"""
    m = max(abs(lo), abs(hi))
    u = np.linspace(-1, 1, B + 1)
    return np.sign(u) * (np.power(1 + mu, np.abs(u)) - 1) / mu * m

def quantize(a, edges):
    """返回 (bin 下标, 反量化重建值)。"""
    idx = np.clip(np.searchsorted(edges, a, side='right') - 1, 0, len(edges) - 2)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return idx, centers[idx]

B = 256
SCHEMES = [('uniform', uniform_edges(-STEER_MAX, STEER_MAX, B)),
           ('quantile', quantile_edges(A, B)),
           ('mu-law', mulaw_edges(-STEER_MAX, STEER_MAX, B))]

print(f"{'分箱':<10s} {'全局 RMS':>10s} {'直行工况 RMS':>14s} {'大转角 RMS':>12s} {'实际 bin 数':>11s}")
err_tab = {}
for name, e in SCHEMES:
    _, rec = quantize(A, e)
    err = rec - A
    r_all = float(np.sqrt((err ** 2).mean()))
    r_st = float(np.sqrt((err[straight] ** 2).mean()))
    r_bg = float(np.sqrt((err[~straight] ** 2).mean()))
    err_tab[name] = (r_all, r_st, r_bg)
    print(f'{name:<10s} {r_all:>10.4f} {r_st:>14.4f} {r_bg:>12.4f} {len(e)-1:>11d}')

# 理论值校验：均匀分箱的 RMS = Δ/sqrt(12)
delta = 2 * STEER_MAX / B
theo = delta / math.sqrt(12)
print(f'\n均匀分箱理论 RMS = Δ/√12 = {delta:.4f}/{math.sqrt(12):.3f} = {theo:.4f}°')
assert abs(err_tab['uniform'][0] - theo) < 5e-3, '实测应与 Δ/√12 吻合'
assert err_tab['quantile'][1] < err_tab['uniform'][1] / 10, '分位数分箱在直行工况应好一个数量级以上'
assert err_tab['quantile'][2] > err_tab['uniform'][2] * 5, '代价是大转角工况显著变差'
assert err_tab['mu-law'][1] < err_tab['uniform'][1] and err_tab['mu-law'][2] < err_tab['quantile'][2]
print('\n⚠️  分位数分箱把直行精度提升 26 倍，代价是大转角精度掉 12 倍 —— 掉头这类工况反而变差。')
print('✅ μ-law 是折中：两头都比 uniform 好，且**边界不依赖数据**（不用跟着数据版本走）。')

In [ ]:
# 量化误差会不会伤人？取决于「它被积分多久」
def lateral_offset(delta_deg, arc_m, L=WHEELBASE):
    """小角度下 κ≈δ/L，行驶弧长 arc_m 后的横向偏移 y ≈ κ·arc²/2。"""
    kappa = np.deg2rad(delta_deg) / L
    return kappa * arc_m ** 2 / 2

V_HIGHWAY = 30.0        # m/s ≈ 108 km/h
print(f'1 个 bin 的角度误差 Δ = {delta:.4f}°，对应曲率 {np.deg2rad(delta)/WHEELBASE:.3e} /m\n')
print(f"{'持续时间':>10s} {'行驶距离':>10s} {'横向偏移':>12s}")
for dt_s in [0.1, 0.5, 1.0, 2.0]:
    arc = V_HIGHWAY * dt_s
    print(f'{dt_s:>9.1f}s {arc:>9.1f}m {lateral_offset(delta, arc)*100:>11.2f} cm')

y_01 = lateral_offset(delta, V_HIGHWAY * 0.1)
y_10 = lateral_offset(delta, V_HIGHWAY * 1.0)
assert y_01 < 0.02 and 0.5 < y_10 < 1.0
print(f'\n⚠️  同一个量化误差：0.1 s 内是 {y_01*1000:.1f} mm（可忽略），'
      f'1.0 s 内是 {y_10:.2f} m（超过半车道容差）。')
print('✅ 结论：**离散化的可接受性 = 这个量被积分多久**。')
print('   输出轨迹点 → 误差不积分；输出控制量 → 误差沿时间积分成位置误差。')

## 2 · token 预算：自回归解码是延迟的大头

$K$ 步 chunk × $D$ 维动作 = $K\cdot D$ 个 token，**每个 token 都要过一次完整的 LLM 前向**。

In [ ]:
MS_PER_TOKEN = 25.0        # 车端 2–4B 模型单 token 解码的典型量级

def token_budget(K, D, bins=256, ms_per_token=MS_PER_TOKEN):
    n_tok = K * D
    return dict(tokens=n_tok, bits=n_tok * int(math.log2(bins)),
                decode_ms=n_tok * ms_per_token,
                max_hz=1000.0 / max(n_tok * ms_per_token, 1e-9))

CFGS = [('单步 2 维 (δ, a)', 1, 2), ('8 步轨迹 2 维', 8, 2),
        ('16 步轨迹 3 维 (x,y,v)', 16, 3), ('机器人 7-DoF 16 步', 16, 7)]
print(f"{'配置':<24s} {'tokens':>7s} {'bits':>6s} {'解码 ms':>9s} {'可达频率':>10s} {'评价'}")
for name, K, D in CFGS:
    b = token_budget(K, D)
    verdict = '✅' if b['decode_ms'] <= 100 else ('⚠️' if b['decode_ms'] <= 500 else '❌ 车端不可用')
    print(f"{name:<24s} {b['tokens']:>7d} {b['bits']:>6d} {b['decode_ms']:>9.0f} "
          f"{b['max_hz']:>9.2f}Hz {verdict}")

b48 = token_budget(16, 3)
assert b48['tokens'] == 48 and abs(b48['decode_ms'] - 1200) < 1e-6
assert token_budget(16, 7)['decode_ms'] > 2500
print('\n⚠️  16 步 × 3 维 = 48 个 token = **1.2 秒**光是解码。这不是优化问题，是路线问题。')
print('✅ 三条出路：①并行解码（放弃维度间自回归）②换非自回归动作头 ③降频 + chunking + 分层。')
print('   量产方案基本是 ②+③ 的组合 —— 这也是本模块后面几节的主线。')

## 3 · 多模态动作分布：回归会输出「平均的错误动作」

场景：路口，专家演示里 50% 左转（−30°）、50% 右转（+30°），
**直行方向 $|\theta| < 12°$ 是隔离墩**。用镜像构造让经验分布严格对称。

In [ ]:
BLOCK_DEG = 12.0                      # |θ| < 12° 是隔离墩所在的扇区
def collide(theta):
    return np.abs(np.asarray(theta)) < BLOCK_DEG

def make_intersection(n_half, w_left=0.5, seed=7, sigma=4.0):
    """镜像构造：w_left=0.5 时经验分布**严格对称**，均值与中位数都精确为 0。"""
    r = np.random.default_rng(seed)
    base = r.normal(30.0, sigma, n_half)              # 转弯幅度（度）
    n_left = int(round(2 * n_half * w_left))
    n_right = 2 * n_half - n_left
    def take(k, sign):
        if k <= n_half:
            return sign * base[:k]
        return sign * np.concatenate([base, r.normal(30.0, sigma, k - n_half)])
    return np.concatenate([take(n_left, -1.0), take(n_right, +1.0)])

theta = make_intersection(10_000)
print(f'专家动作: N={len(theta)}  左转 {(theta<0).mean():.1%}  右转 {(theta>0).mean():.1%}')
print(f'  经验均值   (= ℓ2 回归的最优解) = {theta.mean():+.6f}°   撞墩 = {collide(theta.mean())}')
print(f'  经验中位数 (= ℓ1 回归的最优解) = {np.median(theta):+.6f}°   撞墩 = {collide(np.median(theta))}')
print(f'  专家动作里落在隔离墩扇区的比例 = {collide(theta).mean():.4%}')

assert abs(theta.mean()) < 1e-9 and abs(np.median(theta)) < 1e-9
assert collide(theta.mean()) and collide(np.median(theta))
assert collide(theta).mean() < 0.001, '专家从来不走中间'
print('\n★ **专家 0% 走中间，而两种回归都 100% 输出中间。**')

In [ ]:
# 用梯度下降验证：ℓ2 常数预测器确实收敛到均值，而不是某个 mode
def gd_fit(theta, loss='l2', w0=25.0, lr=0.05, steps=4000):
    w = w0
    for _ in range(steps):
        if loss == 'l2':
            g = 2.0 * (w - theta).mean()
        else:                                  # ℓ1 的次梯度
            g = np.sign(w - theta).mean()
        w -= lr * g
    return w

w_l2 = gd_fit(theta, 'l2', w0=25.0)     # 从「右转」附近出发
w_l2b = gd_fit(theta, 'l2', w0=-25.0)   # 从「左转」附近出发
print(f'ℓ2 GD 从 +25° 出发 → {w_l2:+.4f}°')
print(f'ℓ2 GD 从 −25° 出发 → {w_l2b:+.4f}°   ← **两个初值收敛到同一点**')
assert abs(w_l2) < 0.05 and abs(w_l2b) < 0.05
print('\n⚠️  这不是「陷入局部最优」——ℓ2 的最优解本来就唯一，且就是那个错答案。')

# 容量无关：给 32 维（与左右无关的）特征也没用
N = len(theta)
X = np.hstack([np.ones((N, 1)), np.random.default_rng(1).normal(size=(N, 32))])
beta = np.linalg.lstsq(X, theta, rcond=None)[0]
pred = X @ beta
print(f'\n加 32 维特征的最小二乘：预测均值 {pred.mean():+.4f}°，预测标准差 {pred.std():.3f}°，'
      f'撞墩率 {collide(pred).mean():.1%}')
assert collide(pred).mean() > 0.99
print('✅ **加容量无效**：只要特征区分不出「这次该左还是该右」，最优解就还是条件均值。')

In [ ]:
# ℓ1 不是解药：它对混合权重是「刀锋式」的
print(f"{'w_left':>7s} {'均值':>9s} {'撞墩':>6s} {'中位数':>10s} {'撞墩':>6s}")
means, medians = [], []
for w in [0.30, 0.45, 0.49, 0.50, 0.51, 0.55, 0.70]:
    t = make_intersection(10_000, w_left=w)
    m_, md_ = float(t.mean()), float(np.median(t))
    means.append(m_); medians.append(md_)
    print(f'{w:>7.2f} {m_:>+9.2f} {str(collide(m_)):>6s} {md_:>+10.2f} {str(collide(md_)):>6s}')

assert all(collide(m) for m in means), 'w∈[0.3,0.7] 内均值**始终**落在隔离墩里'
i49, i51 = 2, 4
assert medians[i49] > 20 and medians[i51] < -20
assert abs(medians[i49] - medians[i51]) > 40
print(f'\n⚠️  中位数在 w=0.49→0.51 之间跳了 {abs(medians[i49]-medians[i51]):.1f}°（+21.7 → −21.7）。')
print('    数据里 2% 的比例变化，让决策 180° 翻转 —— **ℓ1 只是把「平均错」换成了「不稳定」**。')
print('✅ 两种失败模式：ℓ2 稳定地错，ℓ1 不稳定地对。都不能上车。')

In [ ]:
# 三种「补救」全部无效 vs 一个真正能表达分布的头
# ① 回归 + 事后加噪声
noisy = theta.mean() + np.random.default_rng(2).normal(0, theta.std(), 20_000)
# ② 回归 + 加大容量（上面已证）
# ③ 分类头：把动作空间分箱，学一个 **完整的分布**
NB_ = 41
edges_c = np.linspace(-45, 45, NB_ + 1)
centers_c = 0.5 * (edges_c[:-1] + edges_c[1:])
idx_c = np.clip(np.searchsorted(edges_c, theta, side='right') - 1, 0, NB_ - 1)
p_cls = np.bincount(idx_c, minlength=NB_).astype(float)
p_cls /= p_cls.sum()
cls_argmax = float(centers_c[p_cls.argmax()])
cls_sample = np.random.default_rng(3).choice(centers_c, size=20_000, p=p_cls)

print(f"{'方案':<28s} {'输出示例':>12s} {'撞墩率':>9s}")
print(f"{'ℓ2 回归（点估计）':<28s} {theta.mean():>+11.2f}° {1.0:>8.1%}")
print(f"{'ℓ2 回归 + 事后加高斯噪声':<28s} {'~N(0,30)':>12s} {collide(noisy).mean():>8.1%}")
print(f"{'分类头 argmax':<28s} {cls_argmax:>+11.2f}° {float(collide(cls_argmax)):>8.1%}")
print(f"{'分类头按概率采样':<28s} {'两峰':>12s} {collide(cls_sample).mean():>8.1%}")

assert 0.2 < collide(noisy).mean() < 0.45, '加噪声只是把错答案摊开，仍有大量样本撞墩'
assert not collide(cls_argmax)
assert collide(cls_sample).mean() < 0.005
print('\n⚠️  **在点估计上加噪声 ≠ 建模分布**：得到的是「以错答案为中心的一团」，仍有 ~31% 撞墩。')
print('✅ 采样必须发生在一个真正建模了分布的对象上（softmax / 扩散 / 混合密度），而不是事后抖动。')

## 4 · 流匹配采样器：**1 步 Euler = 回归**

流匹配学的是速度场 $u_t(a)=\mathbb{E}[a_1-a_0\mid a_t=a]$，推理时把 ODE 从 $t{=}0$ 积到 $t{=}1$。
对**高斯混合的目标 + 高斯源**，这个速度场有闭式解，所以我们可以不训练网络就把整条曲线跑出来。

对单个高斯分量（源 $\mathcal N(0,\sigma_0^2)$、目标 $\mathcal N(\mu,\sigma^2)$、线性路径）：
$$u_t^{(k)}(a)=\mu_k+\frac{t\sigma_k^2-(1-t)\sigma_0^2}{(1-t)^2\sigma_0^2+t^2\sigma_k^2}\bigl(a-t\mu_k\bigr)$$
混合分量按后验责任度 $r_k(a,t)$ 加权。

In [ ]:
MU_K = np.array([-30.0, 30.0])       # 两个 mode：左转 / 右转
SIG_K = np.array([4.0, 4.0])
W_K = np.array([0.5, 0.5])
SIG0 = 20.0                          # 源分布 N(0, 20²)

def velocity(a, t):
    """高斯混合目标下的**闭式**边际速度场。a: (n,) -> (n,)"""
    a = np.atleast_1d(a)[:, None]                          # (n,1)
    m = t * MU_K[None, :]                                  # (1,K)
    s2 = (1 - t) ** 2 * SIG0 ** 2 + t ** 2 * SIG_K[None, :] ** 2
    logr = -0.5 * (a - m) ** 2 / s2 - 0.5 * np.log(2 * np.pi * s2) + np.log(W_K[None, :])
    logr -= logr.max(axis=1, keepdims=True)
    r = np.exp(logr); r /= r.sum(axis=1, keepdims=True)     # 后验责任度
    uk = MU_K[None, :] + (t * SIG_K[None, :] ** 2 - (1 - t) * SIG0 ** 2) / s2 * (a - m)
    return (r * uk).sum(axis=1)

def sample_flow(n, steps, seed=3):
    r = np.random.default_rng(seed)
    a = r.normal(0.0, SIG0, n)
    dt = 1.0 / steps
    for i in range(steps):
        a = a + velocity(a, i * dt) * dt                    # 前向 Euler
    return a

# 手算校验：t=0 时各分量责任度 = 先验权重，于是 u_0(a) = E[a_1] − a
u0 = velocity(np.array([7.3, -12.0, 0.0]), 0.0)
assert np.allclose(u0, MU_K @ W_K - np.array([7.3, -12.0, 0.0])), u0
print('u_0(a) = E[a₁] − a =', np.round(u0, 4), '  ← 一步 Euler 后 a ← a + (E[a₁]−a) = E[a₁]')
print('✅ 速度场就位（闭式，无需训练）')

In [ ]:
print(f"{'积分步数':>8s} {'撞墩率':>9s} {'左峰占比':>9s} {'右峰占比':>9s} {'样本 std':>9s} {'动作头前向次数':>13s}")
coll = []
for s in [1, 2, 3, 4, 8, 16, 32]:
    x = sample_flow(20_000, s)
    c = float(collide(x).mean())
    coll.append(c)
    print(f'{s:>8d} {c:>8.1%} {float((x<-BLOCK_DEG).mean()):>8.1%} '
          f'{float((x>BLOCK_DEG).mean()):>8.1%} {x.std():>9.2f} {s:>13d}')

x1 = sample_flow(2000, 1)
assert np.allclose(x1, 0.0, atol=1e-9), '1 步 Euler 必须精确输出条件期望'
assert coll[0] == 1.0, '所以 1 步流匹配 = 回归 = 100% 撞墩'
assert all(coll[i] >= coll[i + 1] - 1e-9 for i in range(len(coll) - 1)), '撞墩率应随步数单调下降'
assert coll[-1] < 0.005

x16 = sample_flow(20_000, 16)
lo, hi = x16[x16 < 0].mean(), x16[x16 > 0].mean()
print(f'\n16 步的两个峰: {lo:+.2f}° / {hi:+.2f}°  （真值 ±30.0°），左右比例 '
      f'{(x16<0).mean():.1%} / {(x16>0).mean():.1%}')
assert abs(lo + 30) < 1.5 and abs(hi - 30) < 1.5
assert 0.45 < (x16 < 0).mean() < 0.55
print('\n★ **1 步 Euler 的输出精确等于条件期望**（样本 std = 0.00）——也就是退化成了回归。')
print('✅ 所以「积分步数」就是多模态表达力的旋钮：1 步没有、2 步 23.5% 撞、4 步 1.7%、16 步 0%。')
print('⚠️  蒸馏到 1–2 步很诱人，但它正好把好不容易得到的多模态能力又丢掉 ——')
print('    而且这个退化在「平均轨迹误差」上几乎看不出来，只在多解场景的分桶评测里暴露。')

## 5 · 轨迹参数化：阶数、参数量，以及「输出控制量」的车型耦合

In [ ]:
DT_TRAJ = 0.2                     # 轨迹点间隔
T_STEPS = 20                      # 20 点 × 0.2 s = 4 秒视野

def rollout(delta_seq, L, v=V_HIGHWAY, dt=DT_TRAJ):
    """自行车模型：给一串前轮转角，积分出轨迹。"""
    x = y = psi = 0.0
    xs, ys = [], []
    for d in delta_seq:
        kappa = math.tan(d) / L
        x += v * math.cos(psi) * dt
        y += v * math.sin(psi) * dt
        psi += v * kappa * dt
        xs.append(x); ys.append(y)
    return np.array(xs), np.array(ys)

t_axis = np.arange(T_STEPS) * DT_TRAJ
delta_seq = np.deg2rad(0.25) * np.sin(2 * np.pi * t_axis / 4.0)     # 一次变道式转角序列

print('★ 同一串**控制量**，换车型会怎样：')
print(f"{'轴距 L':>8s} {'4 s 后横向位移':>16s} {'相对 L=2.7 的偏差':>18s}")
ref_y = None
devs = []
for L in [2.7, 2.9, 3.1]:
    xs, ys = rollout(delta_seq, L)
    if ref_y is None:
        ref_y = ys[-1]
    devs.append(ys[-1] - ref_y)
    print(f'{L:>7.1f}m {ys[-1]:>15.3f}m {ys[-1]-ref_y:>+17.3f}m')

assert abs(devs[-1]) > 0.4, '轴距差 0.4 m 应造成 >0.4 m 的横向偏差'
print(f'\n⚠️  轴距从 2.7 变到 3.1 m（同一平台的不同车型很常见），'
      f'4 秒后横向差 {abs(devs[-1]):.2f} m —— 超过 1/8 车道宽。')
print('✅ 输出**控制量** = 把车辆动力学烧进权重 → 换车型/换轮胎/满载空载都要重采数据重训。')
print('✅ 输出**轨迹** = 动力学被隔离在下游控制器里 → 那是一份按车型配置的代码，不是一份权重。')

In [ ]:
# 参数化的表达力-参数量取舍
xs, ys = rollout(delta_seq, 2.9)
y_dbl = 1.8 * np.sin(2 * np.pi * xs / 60.0)      # 双变道（绕障）——形状更复杂

def fit_err(x, y, deg):
    return float(np.abs(np.polyval(np.polyfit(x, y, deg), x) - y).max())

print(f"{'参数化':<18s} {'参数量':>7s} {'单变道最大误差':>15s} {'双变道最大误差':>15s}")
for deg in [3, 5, 7, 9]:
    print(f'{f"{deg} 次多项式":<18s} {deg+1:>7d} {fit_err(xs,ys,deg):>14.4f}m '
          f'{fit_err(xs,y_dbl,deg):>14.4f}m')
print(f'{"原始 waypoints":<18s} {2*len(xs):>7d} {0.0:>14.4f}m {0.0:>14.4f}m')

assert 0.02 < fit_err(xs, ys, 3) < 0.2, '三次多项式在单变道上误差在厘米量级'
assert fit_err(xs, ys, 5) < 0.01
assert fit_err(xs, y_dbl, 3) > 1.0, '同样是三次，双变道误差直接到米级'
assert fit_err(xs, y_dbl, 3) > fit_err(xs, y_dbl, 7) * 10, '复杂形状对阶数极敏感'
print('\n⚠️  同一个三次多项式（4 个参数）：单变道误差 7.9 cm（可接受），'
      '双变道 1.65 m（完全不可用）。')
print('✅ **参数化的够不够，取决于要表达的轨迹形状复杂度，而不是取决于视野长度**。')
print('   五次够单变道；绕障/双变道要 7–9 次或分段样条；')
print('   waypoints 用 40 个参数换「零表达力损失 + 可直接逐点校验」，量产多数选它。')

In [ ]:
# 轨迹的**廉价校验**：这是它作为中间表示的核心价值
KAPPA_MAX, A_MAX, JERK_MAX = 0.12, 3.0, 4.0     # 1/m, m/s², m/s³

def check_trajectory(xs, ys, dt=DT_TRAJ, v=V_HIGHWAY, v_limit=None):
    """纯几何/运动学校验，**不需要任何模型**，微秒级。"""
    dx, dy = np.gradient(xs, dt), np.gradient(ys, dt)
    ddx, ddy = np.gradient(dx, dt), np.gradient(dy, dt)
    speed = np.hypot(dx, dy)
    kappa = np.abs(dx * ddy - dy * ddx) / np.maximum(speed ** 3, 1e-9)
    a_lat = kappa * speed ** 2
    jerk = np.abs(np.gradient(a_lat, dt))
    rep = {'max_kappa': float(kappa.max()), 'max_a_lat': float(a_lat.max()),
           'max_jerk': float(jerk.max()), 'max_speed': float(speed.max())}
    rep['ok'] = (rep['max_kappa'] <= KAPPA_MAX and rep['max_a_lat'] <= A_MAX
                 and rep['max_jerk'] <= JERK_MAX
                 and (v_limit is None or rep['max_speed'] <= v_limit))
    return rep

ok_rep = check_trajectory(xs, ys)
xs_b, ys_b = rollout(np.deg2rad(6.0) * np.sin(2 * np.pi * t_axis / 4.0), 2.9)   # 过激轨迹
bad_rep = check_trajectory(xs_b, ys_b)
print(f"{'轨迹':<14s} {'max κ':>9s} {'max a_lat':>11s} {'max jerk':>10s} {'通过'}")
for name, r in [('正常变道', ok_rep), ('过激变道', bad_rep)]:
    print(f"{name:<14s} {r['max_kappa']:>9.4f} {r['max_a_lat']:>11.3f} "
          f"{r['max_jerk']:>10.3f} {'✅' if r['ok'] else '❌'}")
assert ok_rep['ok'] and not bad_rep['ok']

# 加上 TSR 给的限速约束（模块 03/04 的接口在这里落地）
r_lim = check_trajectory(xs, ys, v_limit=25.0)      # 前方限速牌 90 km/h = 25 m/s
print(f"\n叠加 TSR 限速 25 m/s 后: max_speed={r_lim['max_speed']:.2f} m/s -> "
      f"{'✅' if r_lim['ok'] else '❌ 违反限速，需重规划或减速'}")
assert not r_lim['ok']
print('\n✅ **轨迹能被廉价校验，控制量不能** —— 给你一串方向盘角度，你得先用车辆模型')
print('   积分成轨迹才知道安不安全，于是动力学又被引回来了。')
print('   「输出一个可被廉价校验的中间表示」是安全关键系统的通用设计原则。')

## 6 · Action chunking 与 temporal ensembling

建模关键：**每一次推理调用都有一个整体偏置 $b_c$**（这次前向的「主观判断」略有不同），
再叠加随预测步数增长的噪声 $\sigma_j = \sigma_0(1+\alpha j)$。
$b_c$ 正是 chunk 边界跳变的来源。

In [ ]:
T_SIM = 240
tt = np.arange(T_SIM)
a_true = 10 * np.sin(2 * np.pi * tt / 60) + 3 * np.sin(2 * np.pi * tt / 17)   # 专家动作序列

def make_preds(K, H, sig_b=0.8, s0=0.15, alpha=0.25, seed=5):
    """每 H 步调用一次模型，一次产出 K 步预测。"""
    r = np.random.default_rng(seed)
    preds = {}
    for c in range(0, T_SIM, H):
        j = np.arange(K)
        b = r.normal(0, sig_b)                               # **本次调用的整体偏置**
        preds[c] = a_true[np.clip(c + j, 0, T_SIM - 1)] + b + r.normal(0, 1, K) * (s0 * (1 + alpha * j))
    return preds

def exec_naive(preds, K, H):
    out = np.zeros(T_SIM)
    for c in sorted(preds):
        for j in range(H):
            if c + j < T_SIM:
                out[c + j] = preds[c][j]
    return out

def exec_ensemble(preds, K, H, m=0.10):
    """ACT 式时序集成：同一时刻被多个 chunk 预测到，按 exp(-m·j) 加权平均（越新鲜权重越大）。"""
    num = np.zeros(T_SIM); den = np.zeros(T_SIM)
    for c in sorted(preds):
        for j in range(K):
            if c + j < T_SIM:
                w = math.exp(-m * j)
                num[c + j] += w * preds[c][j]; den[c + j] += w
    return num / np.maximum(den, 1e-9)

jerk = lambda a: float(np.abs(np.diff(a, 2)).mean())
maxjump = lambda a: float(np.abs(np.diff(a)).max())
rmse = lambda a: float(np.sqrt(((a - a_true) ** 2).mean()))

print(f"{'方案':<26s} {'调用次数':>8s} {'RMSE':>8s} {'平均|Δ²a|':>10s} {'最大|Δa|':>9s}")
res = {}
for name, K, H, ens in [('K=H=1 逐步预测', 1, 1, False), ('K=H=16 无重叠', 16, 16, False),
                        ('K=16,H=4 重叠取最新', 16, 4, False), ('K=16,H=4 + 时序集成', 16, 4, True)]:
    p = make_preds(K, H)
    a = exec_ensemble(p, K, H) if ens else exec_naive(p, K, H)
    res[name] = a
    print(f'{name:<26s} {T_SIM//H:>8d} {rmse(a):>8.4f} {jerk(a):>10.4f} {maxjump(a):>9.4f}')
print(f'{"（专家真值本身）":<26s} {"—":>8s} {0.0:>8.4f} {jerk(a_true):>10.4f} {maxjump(a_true):>9.4f}')

In [ ]:
step = res['K=H=1 逐步预测']; nochunk = res['K=H=16 无重叠']
naive4 = res['K=16,H=4 重叠取最新']; ens4 = res['K=16,H=4 + 时序集成']

# ① chunking 改善时序一致性：逐步预测每步换一次「主观判断」，抖得最厉害
assert jerk(step) > jerk(nochunk) * 1.5, 'chunking 应显著降低抖动'
# ② 重叠取最新（H<K）比无重叠好：只用最新鲜的几步
assert rmse(naive4) < rmse(nochunk) and jerk(naive4) < jerk(nochunk)
# ③ 时序集成再降一档
assert rmse(ens4) < rmse(naive4) * 0.8, 'ensembling 应把 RMSE 再降 20% 以上'
assert jerk(ens4) < jerk(naive4) * 0.8
assert maxjump(ens4) < maxjump(naive4)

print(f'逐步预测 → K=H=16 : jerk {jerk(step):.4f} → {jerk(nochunk):.4f} '
      f'（降 {1-jerk(nochunk)/jerk(step):.0%}，且调用次数少 16 倍）')
print(f'H=16 → H=4 重叠   : RMSE {rmse(nochunk):.4f} → {rmse(naive4):.4f}')
print(f'重叠 → +时序集成   : RMSE {rmse(naive4):.4f} → {rmse(ens4):.4f} '
      f'（降 {1-rmse(ens4)/rmse(naive4):.0%}）, jerk {jerk(naive4):.4f} → {jerk(ens4):.4f}')
print('\n✅ chunking **同时**降低了调用频率和抖动 —— 这两个收益通常是互相冲突的，这里不冲突。')
print('   原因：逐步预测时每一步都换一次「本次前向的主观判断」，chunk 内则天然自洽。')

In [ ]:
# ⚠️ temporal ensembling 的危险边界：**只能在同一个 mode 内平均**
K, H = 8, 4
turn_left = np.full(K, -30.0)          # 上一个 chunk：决定左转
turn_right = np.full(K, +30.0)         # 这一个 chunk：改主意，右转
w = np.exp(-0.10 * np.arange(K))
blend = (w[H] * turn_left[H] + w[0] * turn_right[0]) / (w[H] + w[0])
print(f'上一个 chunk 说 {turn_left[0]:+.0f}°，这一个说 {turn_right[0]:+.0f}°')
print(f'时序集成的结果 = {blend:+.2f}°   撞墩 = {collide(blend)}')
assert collide(blend), '跨 mode 的平均会落进隔离墩'

def mode_switched(prev_chunk, new_chunk, offset, thresh=15.0):
    """用「重叠段的终点距离」判断模型是否换了 mode。"""
    ov = min(len(prev_chunk) - offset, len(new_chunk))
    return float(np.abs(prev_chunk[offset:offset+ov] - new_chunk[:ov]).max()) > thresh

sw = mode_switched(turn_left, turn_right, H)
same = mode_switched(turn_left, turn_left + 1.0, H)
print(f'\nmode 切换检测: 左→右 {sw}   左→左(+1°) {same}')
assert sw and not same
print('✅ 正确做法：先判 mode 是否切换 —— 同 mode 才平均，换 mode 直接切换（并回传为难例）。')
print('⚠️  这是 temporal ensembling 在自动驾驶里最危险的失败模式，机器人上不明显（代价小）。')

## 7 · $(K, H, \Delta t)$ 的四约束求解器

① 视野 $K\Delta t \ge T_{\text{horizon}}$　② 吞吐 $H\Delta t \ge T_{\text{infer}}+T_{\text{comm}}$
③ 反应 $T_{\text{sense}}+T_{\text{infer}}+H\Delta t+T_{\text{act}} \le T_{\text{budget}}$　④ 鲁棒 $(K-H)\Delta t \ge T_{\text{overrun}}$

In [ ]:
EPS = 1e-9        # 浮点保护：0.44/0.01 在二进制里是 43.99999...，不加会误判不可行

def plan_chunk(dt, T_sense, T_infer, T_comm, T_act, T_budget, T_horizon, T_overrun):
    H_min = math.ceil((T_infer + T_comm) / dt - EPS)                          # 约束②
    H_max = math.floor((T_budget - T_sense - T_infer - T_act) / dt + EPS)     # 约束③
    out = {'H_min': H_min, 'H_max': H_max, 'feasible': (H_min <= H_max and H_max >= 1)}
    if out['feasible']:
        H = H_min                                                             # 取最小 H = 反应最快
        K = max(math.ceil(T_horizon / dt - EPS),
                H + math.ceil(T_overrun / dt - EPS))                          # 约束①④
        out.update(H=H, K=K, call_hz=1.0 / (H * dt),
                   T_react=T_sense + T_infer + H * dt + T_act,
                   stale_mean=T_infer + H * dt / 2, stale_max=T_infer + H * dt,
                   overlap=1 - H / K, util=(T_infer + T_comm) / (H * dt))
    return out

BASE = dict(dt=0.2, T_sense=0.06, T_infer=0.25, T_comm=0.03, T_act=0.10,
            T_budget=1.0, T_horizon=3.2, T_overrun=0.4)
r = plan_chunk(**BASE)
print('基线配置求解结果:')
for k in ['H_min', 'H_max', 'H', 'K', 'call_hz', 'T_react', 'stale_mean', 'stale_max',
          'overlap', 'util']:
    print(f'  {k:<12s} {r[k]:.4g}')
print(f"\n  30 m/s 下：平均陈旧距离 {V_HIGHWAY*r['stale_mean']:.1f} m，"
      f"最坏反应距离 {V_HIGHWAY*r['T_react']:.1f} m")

assert r['feasible'] and r['H_min'] == r['H_max'] == 2, 'H 被夹到唯一解 2'
assert r['K'] == 16 and abs(r['call_hz'] - 2.5) < 1e-9
assert abs(r['stale_mean'] - 0.45) < 1e-9 and abs(r['T_react'] - 0.81) < 1e-9
print('\n⚠️  约束②给 H≥2、约束③给 H≤2 —— **可行窗口只剩一个整数点**。')
print('✅ K 由「视野」主导（16 > 4），重叠 87.5% 是免费得到的鲁棒性。')

In [ ]:
# 扫描 T_infer：窗口什么时候变空？
print(f"{'T_infer':>8s} {'H 窗口':>10s} {'H':>4s} {'K':>4s} {'调用 Hz':>8s} {'反应 s':>8s} {'利用率':>7s}")
for ti in [0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45]:
    b = dict(BASE); b['T_infer'] = ti
    rr = plan_chunk(**b)
    win = f"[{rr['H_min']},{rr['H_max']}]"
    if rr['feasible']:
        print(f"{ti:>8.2f} {win:>10s} {rr['H']:>4d} {rr['K']:>4d} {rr['call_hz']:>8.2f} "
              f"{rr['T_react']:>8.2f} {rr['util']:>6.0%}")
    else:
        print(f'{ti:>8.2f} {win:>10s} {"—":>4s} {"—":>4s} {"—":>8s} {"—":>8s}  ❌ 不可行')

assert plan_chunk(**{**BASE, 'T_infer': 0.30})['feasible']
assert not plan_chunk(**{**BASE, 'T_infer': 0.45})['feasible']

# 把约束②③相加消掉 H·Δt，得到一个**与 K、H、Δt 全都无关**的可行性条件
def max_affordable_infer(T_sense, T_comm, T_act, T_budget):
    return (T_budget - T_sense - T_comm - T_act) / 2.0

lim = max_affordable_infer(BASE['T_sense'], BASE['T_comm'], BASE['T_act'], BASE['T_budget'])
print(f'\n★ 连续条件: 2·T_infer + T_sense + T_comm + T_act ≤ T_budget  ⟹  T_infer ≤ {lim:.3f} s')
print('  **推理耗时在反应延迟预算里被算了两次** —— 一次是它自己的延迟，')
print('  一次是「chunk 执行窗口必须长到够跑完一次推理」。')
assert abs(lim - 0.405) < 1e-9
# 整数 H 让实际上限更严；把 Δt 变小可以逼近连续上限
fine = plan_chunk(**{**BASE, 'dt': 0.01, 'T_infer': 0.40})
coarse = plan_chunk(**{**BASE, 'dt': 0.20, 'T_infer': 0.40})
print(f"\nT_infer=0.40 s: Δt=0.20 → {'可行' if coarse['feasible'] else '**不可行**'}（H 只能取整数）; "
      f"Δt=0.01 → {'可行' if fine['feasible'] else '不可行'}（H={fine.get('H')}）")
assert not coarse['feasible'] and fine['feasible']
print('✅ 所以「模型快一倍」对反应延迟的收益是**双倍**的；而推理慢时把 chunk 调短根本无解。')

## ✏️ 练习 1：分箱精度的正反解

实现两个函数：
- `quant_rms(lo, hi, B)` → 均匀分箱的量化 RMS 误差 $\Delta/\sqrt{12}$
- `bins_needed(lo, hi, target_rms)` → 达到目标 RMS 所需的**最少 bin 数**（向上取整，用 `1e-9` 做浮点保护）

In [ ]:
def quant_rms(lo, hi, B):
    # TODO
    raise NotImplementedError

def bins_needed(lo, hi, target_rms):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
r256 = quant_rms(-35.0, 35.0, 256)
assert abs(r256 - 70.0 / 256 / math.sqrt(12)) < 1e-12, r256
assert abs(r256 - 0.078934) < 1e-5
assert bins_needed(-35.0, 35.0, r256) == 256, bins_needed(-35.0, 35.0, r256)
assert bins_needed(-35.0, 35.0, 0.5) == 41, bins_needed(-35.0, 35.0, 0.5)
assert bins_needed(-35.0, 35.0, r256 / 2) == 512      # 精度翻倍 -> bin 数翻倍（1 bit）
assert quant_rms(0.0, 130.0, 256) > quant_rms(0.0, 130.0, 512)
print(f"{'量':<16s} {'范围':>12s} {'B':>6s} {'RMS':>10s}")
for name, lo, hi, B in [('前轮转角(°)', -35, 35, 256), ('前轮转角(°)', -35, 35, 1024),
                        ('车速(km/h)', 0, 130, 256), ('横向位置(m)', -6, 6, 256)]:
    print(f'{name:<16s} {f"[{lo},{hi}]":>12s} {B:>6d} {quant_rms(lo,hi,B):>10.5f}')
print(f'\n要把转角量化 RMS 压到 0.02°，需要 {bins_needed(-35,35,0.02)} 个 bin '
      f'（= {math.ceil(math.log2(bins_needed(-35,35,0.02)))} bit/维）')
print('✅ 练习 1 通过：**精度每翻一倍要多 1 bit**，而 bit 数直接换算成 token 数与解码延迟。')

## ✏️ 练习 2：点估计 vs 众数

实现 `optimal_point_estimate(samples, loss)`（`loss ∈ {'l2','l1'}`，返回最优常数预测）
与 `mode_estimate(samples, nbins, lo, hi)`（分箱后取概率最大的 bin 的中心）。
用它们在第 3 节的双峰数据上复现「回归撞墩、众数不撞」。

In [ ]:
def optimal_point_estimate(samples, loss='l2'):
    # TODO: 'l2' -> 条件期望；'l1' -> 条件中位数
    raise NotImplementedError

def mode_estimate(samples, nbins=41, lo=-45.0, hi=45.0):
    # TODO: 分箱统计 -> 取质量最大的 bin 的中心
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测（先手算小例子）——
toy = np.array([-10.0, -10.0, -10.0, 30.0])
assert abs(optimal_point_estimate(toy, 'l2') - 0.0) < 1e-12        # 均值 = (-30+30)/4 = 0
assert abs(optimal_point_estimate(toy, 'l1') + 10.0) < 1e-12       # 中位数 = -10（贴到多数派）
m_ = optimal_point_estimate(np.array([1.0, 2.0, 3.0, 100.0]), 'l1')
assert abs(m_ - 2.5) < 1e-12, m_                                   # 中位数不受离群值影响
assert abs(mode_estimate(np.array([-30.]*7 + [30.]*3)) + 30.0) < 3.0

th = make_intersection(10_000)
p2 = optimal_point_estimate(th, 'l2'); p1 = optimal_point_estimate(th, 'l1')
pm = mode_estimate(th)
print(f'ℓ2 最优 = {p2:+.4f}°  撞墩={collide(p2)}')
print(f'ℓ1 最优 = {p1:+.4f}°  撞墩={collide(p1)}')
print(f'众数    = {pm:+.4f}°  撞墩={collide(pm)}')
assert abs(p2) < 1e-9 and abs(p1) < 1e-9 and collide(p2) and collide(p1)
assert abs(abs(pm) - 30.0) < 3.0 and not collide(pm)
print('\n✅ 练习 2 通过：**两种回归都落在两峰中间的低密度区，众数落在峰上。**')
print('   这就是「输出头必须建模分布」的最短证明。')

## ✏️ 练习 3：在延迟预算里选流匹配步数

实现 `pick_flow_steps(step_ms, budget_ms, collision_by_step, max_collision)`：
在 `steps * step_ms <= budget_ms` 且撞墩率 `<= max_collision` 的候选里选**最大**步数，
返回 `(steps, collision)`；一个都不满足时返回 `(None, None)`。

In [ ]:
def pick_flow_steps(step_ms, budget_ms, collision_by_step, max_collision=1.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
COLL = {1: 1.000, 2: 0.235, 3: 0.057, 4: 0.017, 8: 0.002, 16: 0.000, 32: 0.000}
assert pick_flow_steps(8.0, 100.0, COLL) == (8, 0.002)
assert pick_flow_steps(8.0, 20.0, COLL) == (2, 0.235)
assert pick_flow_steps(8.0, 5.0, COLL) == (None, None)          # 连 1 步都跑不完
assert pick_flow_steps(8.0, 100.0, COLL, max_collision=0.001) == (None, None)
assert pick_flow_steps(2.0, 100.0, COLL, max_collision=0.01) == (32, 0.000)
print(f"{'动作头单步 ms':>13s} {'预算 ms':>9s} {'撞墩上限':>9s} {'选用步数':>9s} {'撞墩率':>8s}")
for sm, bd, mc in [(25.0, 100.0, 1.0), (8.0, 100.0, 1.0), (8.0, 100.0, 0.01),
                   (2.0, 100.0, 0.01), (25.0, 20.0, 1.0)]:
    s_, c_ = pick_flow_steps(sm, bd, COLL, mc)
    txt = f'{s_:>9d} {c_:>7.1%}' if s_ else f"{'不可行':>9s} {'—':>8s}"
    print(f'{sm:>13.1f} {bd:>9.1f} {mc:>9.2%}{txt}')
print('\n✅ 练习 3 通过：**动作头做小 = 能多跑几步 = 多模态表达力**。')
print('   25 ms/步的头在 100 ms 预算里只能跑 4 步（撞墩 1.7%）；2 ms/步的头能跑 32 步（0%）。')

## ✏️ 练习 4：chunk 配置的可行性

实现 `chunk_feasible(dt, T_sense, T_infer, T_comm, T_act, T_budget, T_horizon, T_overrun)`，
返回 `(feasible, H, K)`；不可行时 `(False, None, None)`。规则同第 7 节（记得用 `EPS` 做浮点保护）。
再实现 `max_affordable_infer(T_sense, T_comm, T_act, T_budget)`（与 $\Delta t$ 无关的上限）。

In [ ]:
def chunk_feasible(dt, T_sense, T_infer, T_comm, T_act, T_budget, T_horizon, T_overrun):
    # TODO
    raise NotImplementedError

def max_affordable_infer(T_sense, T_comm, T_act, T_budget):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert chunk_feasible(**BASE) == (True, 2, 16), chunk_feasible(**BASE)
assert chunk_feasible(**{**BASE, 'T_infer': 0.10}) == (True, 1, 16)
assert chunk_feasible(**{**BASE, 'T_infer': 0.45}) == (False, None, None)
assert chunk_feasible(**{**BASE, 'T_horizon': 6.0})[2] == 30      # 视野变长 -> K 变大
assert chunk_feasible(**{**BASE, 'T_overrun': 5.0})[2] == 27      # H=2 + ceil(5/0.2)=25
assert abs(max_affordable_infer(0.06, 0.03, 0.10, 1.0) - 0.405) < 1e-9
assert abs(max_affordable_infer(0.06, 0.03, 0.10, 0.6) - 0.205) < 1e-9

print(f"{'反应预算 s':>11s} {'可承受 T_infer':>15s} {'@Δt=0.2 是否可行(T_infer=0.25)':>32s}")
for bud in [0.6, 0.8, 1.0, 1.5]:
    lim_ = max_affordable_infer(0.06, 0.03, 0.10, bud)
    fe = chunk_feasible(**{**BASE, 'T_budget': bud})
    print(f'{bud:>11.2f} {lim_:>15.3f} {str(fe):>32s}')
print('\n✅ 练习 4 通过：**反应延迟预算收紧 0.4 s，可承受的推理时间就砍掉 0.2 s** ——')
print('   因为 T_infer 在预算里被算了两次。这条关系应该直接写进模型选型的输入条件。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def quant_rms(lo, hi, B):
    return (hi - lo) / B / math.sqrt(12)

def bins_needed(lo, hi, target_rms):
    return math.ceil((hi - lo) / (target_rms * math.sqrt(12)) - 1e-9)

In [ ]:
# 练习 2 参考答案
def optimal_point_estimate(samples, loss='l2'):
    a = np.asarray(samples, dtype=float)
    return float(a.mean()) if loss == 'l2' else float(np.median(a))

def mode_estimate(samples, nbins=41, lo=-45.0, hi=45.0):
    a = np.asarray(samples, dtype=float)
    edges = np.linspace(lo, hi, nbins + 1)
    idx = np.clip(np.searchsorted(edges, a, side='right') - 1, 0, nbins - 1)
    counts = np.bincount(idx, minlength=nbins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return float(centers[counts.argmax()])

In [ ]:
# 练习 3 参考答案
def pick_flow_steps(step_ms, budget_ms, collision_by_step, max_collision=1.0):
    ok = [(s, c) for s, c in collision_by_step.items()
          if s * step_ms <= budget_ms and c <= max_collision]
    if not ok:
        return (None, None)
    return max(ok, key=lambda sc: sc[0])

In [ ]:
# 练习 4 参考答案
def chunk_feasible(dt, T_sense, T_infer, T_comm, T_act, T_budget, T_horizon, T_overrun):
    H_min = math.ceil((T_infer + T_comm) / dt - EPS)
    H_max = math.floor((T_budget - T_sense - T_infer - T_act) / dt + EPS)
    if H_min > H_max or H_max < 1:
        return (False, None, None)
    H = H_min
    K = max(math.ceil(T_horizon / dt - EPS), H + math.ceil(T_overrun / dt - EPS))
    return (True, H, K)

def max_affordable_infer(T_sense, T_comm, T_act, T_budget):
    return (T_budget - T_sense - T_comm - T_act) / 2.0

---
## 🧪 真实工程胶囊：动作头选型与上车检查单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# VLA 动作头选型与上车检查单
# ══════════════════════════════════════════════════════════════════════

# ── ① 先回答一个问题：你的 p(a|o) 是单峰还是多峰？ ──────────────────
#   不做这一步就选头 = 赌博。三个可直接跑的诊断：
#   (a) 同场景聚类内专家动作的方差
#       cluster = knn(scene_embedding, k=50)
#       assert action_var_within_cluster < THRESH      # 超了就是多峰
#   (b) 残差直方图：res = a_expert - regressor(o)；双峰 -> 多峰
#   (c) 验证 loss 高原：若 loss 卡在非零平台且加数据不降，
#       那个平台就是 E[Var(a|o)]，**不是模型的错**
#   多峰场景清单（TSR 相关）：路口左右转 / 施工绕行方向 / 限速解除后加不加速 /
#   可变电子牌读数跳变 / 黄灯冲或停

# ── ② 输出什么（参数化） ─────────────────────────────────────────────
#   量产默认：**等时间间隔 waypoints (x, y, v)**，Δt = 0.1~0.2 s，视野 3~5 s
#   不要输出控制量：轴距差 0.4 m 就让同一串转角序列 4 s 后横向差 0.47 m
#   拆两路更省：几何用能表达多模态的头 + 速度剖面用回归（给定几何后通常单峰）

# ── ③ 选头 ──────────────────────────────────────────────────────────
#   单峰   -> 回归 MLP 头（最快最稳，别过度设计）
#   多峰   -> 锚点分类 + 残差回归（1 次前向、可解释）
#            或 流匹配头（表达力最强，N 次小头前向）
#   **不要**用自回归动作 token：16x3 = 48 token x 25 ms = 1.2 s，车端不可用
#   流匹配的关键参数：
#       n_steps: 8            # **固定步数**！自适应步数会让延迟重新依赖数据
#       solver:  euler        # 1 步 = 条件期望 = 退化成回归，别蒸到 1 步
#       action_expert_params: ~300M    # 主干只跑 1 次，只有它跑 n_steps 次

# ── ④ chunk 配置（四约束求解，别拍脑袋）────────────────────────────
#   ① 视野   K*dt      >= T_horizon           (3~5 s，安全层要有东西可校验)
#   ② 吞吐   H*dt      >= T_infer + T_comm
#   ③ 反应   T_sense + T_infer + H*dt + T_act <= T_budget
#   ④ 鲁棒   (K-H)*dt  >= T_overrun
#   ★ ②+③ 消掉 H*dt 得到与 dt 无关的硬条件：
#       2*T_infer + T_sense + T_comm + T_act <= T_budget
#     => T_budget=1.0s, T_sense=60ms, T_comm=30ms, T_act=100ms 时 T_infer <= 405 ms
#     **推理耗时被算了两次**，所以模型提速的收益是双倍的
#   典型解：dt=0.2, H=2, K=16 -> 调用 2.5 Hz, 重叠 87.5%, 最坏反应 0.81 s

# ── ⑤ temporal ensembling 的护栏 ────────────────────────────────────
#   weights = exp(-m * j)          # j = 该预测的「年龄」，m ~ 0.1
#   **必须先判 mode 是否切换**：
#       if max|prev_chunk[H:] - new_chunk[:.]| > MODE_SWITCH_THRESH:
#           use new_chunk directly          # 换 mode -> 切换，**不要平均**
#           log_hard_case(scene)            # 顺便回传为难例（C58 的触发器）
#       else:
#           blend()                          # 同 mode -> 平均，降 jerk
#   跨 mode 平均 = 左转和右转平均成直行 = 撞隔离墩

# ── ⑥ 必做的评测（平均指标看不出多模态塌陷）────────────────────────
#   · 建一个「多解场景」子集（路口 / 绕障 / 限速解除），单独报指标
#   · 报 **mode 覆盖率**：采 N 条轨迹，落在各 mode 上的比例是否与专家一致
#   · 报 **不可行率**：轨迹违反 κ_max / a_max / jerk_max / TSR 限速 的比例
#   · 报 **jerk 与 chunk 边界跳变**（时序一致性）
#   · 蒸馏前后必须对比 mode 覆盖率 —— 塌回单峰在平均轨迹误差上看不出来

# ── ⑦ 红线 ─────────────────────────────────────────────────────────
#   · 动作头之后必须有**无神经网络**的可行性投影 + 约束校验层（< 5 ms）
#   · 轨迹必须带时间戳与有效期，超期自动退化到兜底轨迹（看门狗）
#   · 必须有一条**不经过 VLA** 的独立 AEB 通路（< 100 ms）
#   · 云端大模型**绝不进入实时回路**（只做离线蒸馏与难例标注的教师）
'''
print(RECIPE)
for tok in ['E[Var(a|o)]', 'waypoints', 'n_steps', '2*T_infer', 'MODE_SWITCH_THRESH',
            'mode 覆盖率', 'AEB', '405 ms']:
    assert tok in RECIPE, tok
print('✅ 检查单覆盖：多峰诊断 / 参数化 / 选头 / 四约束 / ensembling 护栏 / 评测 / 红线')

### 小结

- **输出头要给的是分布，不是均值。** $\ell_2$ 的最优解是条件期望、$\ell_1$ 是条件中位数，
  对称双峰下两者都落在两峰中间的低密度区——也就是隔离墩所在。
  **这是损失函数的性质，加数据、加容量、事后加噪声全都无效**（notebook 逐一证伪：
  32 维特征后撞墩率仍 100%，加噪声仍 31%）。判据是「$p(a\mid o)$ 单峰否」，不是「任务难不难」。
- **ℓ1 不是解药**：混合权重从 0.49 到 0.51，中位数跳了 43.4°。ℓ2 稳定地错，ℓ1 不稳定地对。
- **1 步 Euler 的流匹配精确等于条件期望**——所以积分步数就是多模态表达力的旋钮
  （1 步 100% 撞、4 步 1.7%、16 步 0%）。**蒸馏到 1–2 步会塌回单峰，且平均指标看不出来。**
- **离散 token 的真实代价是自回归解码**：16 步 × 3 维 = 48 token = 1.2 s。
  机器人可以，车不行——这是「机器人 VLA 方案不能照搬自驾」最硬的一条。
- **输出轨迹而不是控制量**：轴距 2.7→3.1 m，同一串转角 4 s 后横向差 0.47 m；
  而轨迹能用纯几何在微秒级校验曲率/加速度/jerk/限速——**可被廉价校验是安全关键系统的核心要求**。
- **chunking 同时降低调用频率和抖动**（逐步预测每步换一次「主观判断」，jerk 反而最高）；
  **temporal ensembling 只能在同一个 mode 内做**，换 mode 必须切换而不是平均。
- **$(K,H,\Delta t)$ 被四个约束夹死**，且约束②③相加得到与 $\Delta t$ 无关的硬条件
  $2T_{\text{infer}}+T_{\text{sense}}+T_{\text{comm}}+T_{\text{act}}\le T_{\text{budget}}$——
  **推理耗时在反应延迟预算里被算了两次**（本例上限 405 ms）。

下一站：**模块 03 · TSR 输出如何进入 VLA：接口设计** ——
本模块假定「观测已经给定」，但很多多模态其实来自感知的不确定性；
如果感知不把置信度传下来，再强的动作头也无从表达。